# UrbanHub - Analyse Mobilité Urbaine (Phase 2)

Ce carnet analyse les données capturées en streaming depuis l'API CityBikes.

In [ ]:
print("Tentative d'import des bibliothèques...")
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import os
from folium.plugins import HeatMap

# Style des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
print("Bibliothèques chargées avec succès !")

## 1. Chargement des données streaming

In [14]:
file_path = 'Data/Streaming/mobility_data.csv'
print(f"Vérification du fichier : {file_path}")
if os.path.exists(file_path):
    print("Chargement du CSV en cours...")
    df = pd.read_csv(file_path)
    
    print("Correction du format de date (nettoyage)...")
    # Suppression du 'Z' final s'il y a déjà un offset +00:00 (problème api)
    df['timestamp'] = df['timestamp'].astype(str).str.replace('Z', '', regex=False)
    
    print("Conversion des dates...")
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    
    print(f"Succès ! Données chargées : {len(df)} lignes.")
else:
    print("Fichier de données introuvable. Laissez tourner streaming_ingestion.py quelques minutes !")

Vérification du fichier : Data/Streaming/mobility_data.csv
Chargement du CSV en cours...
Correction du format de date (nettoyage)...
Conversion des dates...
Succès ! Données chargées : 21644 lignes.


## 2. Calcul du taux d'utilisation par station

In [11]:
print("Calcul des statistiques...")
df['total_slots'] = df['bikes_available'] + df['free_slots']
# Éviter division par zéro
df['occupancy_rate'] = df.apply(lambda row: row['bikes_available'] / row['total_slots'] if row['total_slots'] > 0 else 0, axis=1)

print("Recherche des derniers états enregistrés...")
latest_status = df.sort_values('timestamp').groupby('station_id').last().reset_index()

print("\nTop 5 des stations les plus saturées (Plein à 100%) :")
print(latest_status.sort_values('occupancy_rate', ascending=False)[['city', 'station_name', 'occupancy_rate']].head())

print("\nStatistiques globales :")
print(f"Taux d'occupation moyen : {latest_status['occupancy_rate'].mean():.1%}")

Calcul des statistiques...
Recherche des derniers états enregistrés...

Top 5 des stations les plus saturées (Plein à 100%) :
          city                      station_name  occupancy_rate
2927     Paris       Choiseul - Quatre Septembre             1.0
616       Lyon                  JAURÈS / GERLAND             1.0
624   Toulouse              LASCROSSES - LECLERC             1.0
678      Paris  Gare RER les Saules - Chandigarh             1.0
2712  Toulouse              SAINT-SERNIN - MERLY             1.0

Statistiques globales :
Taux d'occupation moyen : 38.5%


## 3. Cartographie des stations critiques (Vides)

In [16]:
print("Filtrage des stations vides...")
empty_stations = latest_status[latest_status['bikes_available'] == 0]
print(f"Nombre de stations vides actuellement : {len(empty_stations)} sur {len(latest_status)}")

print("Génération de la carte interactive...")
m = folium.Map(location=[46.2276, 2.2137], zoom_start=6) # Centré sur la France

for idx, row in empty_stations.head(250).iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='red',
        fill=True,
        fill_opacity=0.7,
        popup=f"<b>{row['city']}</b><br>{row['station_name']}<br>ID: {row['station_id']}<br>ÉTAT: VIDE"
    ).add_to(m)

print("Carte prête !")
m

Filtrage des stations vides...
Nombre de stations vides actuellement : 270 sur 2947
Génération de la carte interactive...
Carte prête !
